In [49]:
from pathlib import Path
from gears import PertData
import pandas as pd
import scanpy as sc
from tqdm import tqdm
import pubchempy as pcp
import re

In [3]:
ref_dir = Path('/Users/djemec/data/jepa/reference_data')
data_dir = Path('/Users/djemec/data/jepa/v0_6')
pert_dir = data_dir / 'pert_embd'
training_dir = data_dir / 'training'
pretraining_dir = data_dir / 'pretraining'

pert_temp = data_dir / 'pert_tmp'

In [5]:
key_ds =  {'k562e':ref_dir / 'k562e'}
datasets= {
    'k562e_raw': ref_dir / 'raw_k562e' / 'K562_essential_raw_singlecell_01.h5ad',
    'rep1e':ref_dir /'rep1e'/'rpe1_raw_singlecell_01.h5ad',
    'k562gw':ref_dir /'k562gw'/'perturb_processed.h5ad',
    'adamson':ref_dir /'adamson'/'AdamsonWeissman2016_GSM2406681_10X010.h5ad',
    'norman':ref_dir /'norman'/'NormanWeissman2019_filtered.h5ad',
    'sciplex':ref_dir /'sciplex'/'SrivatsanTrapnell2020_sciplex3.h5ad',
    
    
}

In [6]:
splits= ['train','val','test']

## Reivew Key Dataset and extract heldout perturations

In [ ]:
dataset_name = 'k562e'
chunk_size = 10000     
n_genes = 8192 # 2**13
count_normalize_target = 1e4 

In [ ]:
pert_data = PertData(key_ds['k562e']) 
pert_data.load(data_name='replogle_k562_essential')

In [ ]:
pert_data.prepare_split(split='simulation', seed=1) 
split_map = pert_data.set2conditions 

In [ ]:
train_pert = set(split_map['train'])
test_pert = set(split_map['test'])
val_pert = set(split_map['val'])

test_pert & train_pert, val_pert & train_pert, test_pert & val_pert

In [ ]:
len(train_pert), len(test_pert), len(val_pert)

### Create unified clean names

In [5]:
def clean_gears_name(name):
    # GEARS format is 'Gene+ctrl' -> We want 'Gene'
    if name.endswith('+ctrl'):
        return name.replace('+ctrl', '')
    if name.startswith('ctrl+'):
        return name.replace('ctrl+', '')
    elif name == 'ctrl':
        return 'control'
    return str.strip(name)

In [ ]:
train_pert = set([clean_gears_name(i) for i in train_pert])
test_pert = set([clean_gears_name(i) for i in test_pert])
val_pert = set([clean_gears_name(i) for i in val_pert])
print(f'train:{len(train_pert)} | test: {len(test_pert)} | val: {len(val_pert)}')

## Extract Pertrubations from other datasets

#### k562 essential

In [7]:
ds_key = 'k562e_raw'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')

obs_df = ds_adata.obs
obs_df.head()

,gem_group,gene,gene_id,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,core_scale_factor,core_adjusted_UMI_count
cell_barcode,,,,,,,,,,,
AAACCCAAGAAATCCA-27,27,NAF1,ENSG00000145414,P1P2,5449_NAF1_P1P2_ENSG00000145414,NAF1_+_164087918.23-P1P2|NAF1_-_164087674.23-P1P2,0.112083,11438.0,0.013047,0.813253,14064.512695
AAACCCAAGAACTTCC-31,31,BUB1,ENSG00000169679,P1P2,935_BUB1_P1P2_ENSG00000169679,BUB1_-_111435363.23-P1P2|BUB1_-_111435372.23-P1P2,0.179895,5342.0,-1.522247,0.844107,6328.584473
AAACCCAAGAAGCCAC-34,34,UBL5,ENSG00000198258,P1P2,9534_UBL5_P1P2_ENSG00000198258,UBL5_-_9938639.23-P1P2|UBL5_+_9938801.23-P1P2,0.105287,17305.0,0.384157,1.091537,15853.792969
AAACCCAAGAATAGTC-43,43,C9orf16,ENSG00000171159,P1P2,1131_C9orf16_P1P2_ENSG00000171159,C9orf16_+_130922603.23-P1P2|C9orf16_+_13092264...,0.099359,30244.0,3.721912,0.948277,31893.619141
AAACCCAAGACAGCGT-28,28,TIMM9,ENSG00000100575,P1P2,8927_TIMM9_P1P2_ENSG00000100575,TIMM9_-_58893843.23-P1P2|TIMM9_-_58893848.23-P1P2,0.137623,8407.0,-0.975371,0.868942,9674.979492


In [8]:
seq_df = pd.read_csv(ref_dir / 'crispr' / 'crispri_seq.csv')
seq_df.set_index('cell_barcode', inplace=True)
seq_df.head()

,gene,gem_group,gene_id,UMI_count,sgID_AB,guide_seq_a,guide_seq_b
cell_barcode,,,,,,,
AAACCCAAGAAATCCA-27,NAF1,27,ENSG00000145414,11438.0,NAF1_+_164087918.23-P1P2|NAF1_-_164087674.23-P1P2,GGAGCCGTGAGCTTGTCCAG,GCCGCGACGGCGTTCAGAAC
AAACCCAAGAACTTCC-31,BUB1,31,ENSG00000169679,5342.0,BUB1_-_111435363.23-P1P2|BUB1_-_111435372.23-P1P2,GGACAAGCGCCGGGCCTCAG,GCGGGCCTCAGCGGAACCCA
AAACCCAAGAAGCCAC-34,UBL5,34,ENSG00000198258,17305.0,UBL5_-_9938639.23-P1P2|UBL5_+_9938801.23-P1P2,GGGTGAGGAGCTGGTGGCGT,GCCCAGGGCCGCGAACCCCG
AAACCCAAGAATAGTC-43,C9orf16,43,ENSG00000171159,30244.0,C9orf16_+_130922603.23-P1P2|C9orf16_+_13092264...,GGCCGGCGCCGGATGGAAGG,GGCCGCGCGACGATGGAACG
AAACCCAAGACAGCGT-28,TIMM9,28,ENSG00000100575,8407.0,TIMM9_-_58893843.23-P1P2|TIMM9_-_58893848.23-P1P2,GGGGACGGTTGAGCCTTGGG,GGGTTGAGCCTTGGGAGGGA


In [9]:
common_idx = obs_df.index.intersection(seq_df.index)
seq_df = seq_df.loc[common_idx]

In [10]:
cols_to_add = ['gene','gene_id','gem_group', 'sgID_AB', 'guide_seq_a', 'guide_seq_b']
for c in cols_to_add:
    obs_df[c] = seq_df[c].values

obs_df.head()

,gem_group,gene,gene_id,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,core_scale_factor,core_adjusted_UMI_count,guide_seq_a,guide_seq_b
cell_barcode,,,,,,,,,,,,,
AAACCCAAGAAATCCA-27,27,NAF1,ENSG00000145414,P1P2,5449_NAF1_P1P2_ENSG00000145414,NAF1_+_164087918.23-P1P2|NAF1_-_164087674.23-P1P2,0.112083,11438.0,0.013047,0.813253,14064.512695,GGAGCCGTGAGCTTGTCCAG,GCCGCGACGGCGTTCAGAAC
AAACCCAAGAACTTCC-31,31,BUB1,ENSG00000169679,P1P2,935_BUB1_P1P2_ENSG00000169679,BUB1_-_111435363.23-P1P2|BUB1_-_111435372.23-P1P2,0.179895,5342.0,-1.522247,0.844107,6328.584473,GGACAAGCGCCGGGCCTCAG,GCGGGCCTCAGCGGAACCCA
AAACCCAAGAAGCCAC-34,34,UBL5,ENSG00000198258,P1P2,9534_UBL5_P1P2_ENSG00000198258,UBL5_-_9938639.23-P1P2|UBL5_+_9938801.23-P1P2,0.105287,17305.0,0.384157,1.091537,15853.792969,GGGTGAGGAGCTGGTGGCGT,GCCCAGGGCCGCGAACCCCG
AAACCCAAGAATAGTC-43,43,C9orf16,ENSG00000171159,P1P2,1131_C9orf16_P1P2_ENSG00000171159,C9orf16_+_130922603.23-P1P2|C9orf16_+_13092264...,0.099359,30244.0,3.721912,0.948277,31893.619141,GGCCGGCGCCGGATGGAAGG,GGCCGCGCGACGATGGAACG
AAACCCAAGACAGCGT-28,28,TIMM9,ENSG00000100575,P1P2,8927_TIMM9_P1P2_ENSG00000100575,TIMM9_-_58893843.23-P1P2|TIMM9_-_58893848.23-P1P2,0.137623,8407.0,-0.975371,0.868942,9674.979492,GGGGACGGTTGAGCCTTGGG,GGGTTGAGCCTTGGGAGGGA


#### rep1e

In [11]:
ds_key = 'rep1e'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')

obs_df = ds_adata.obs
obs_df.head()

,gem_group,gene,gene_id,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,core_scale_factor,core_adjusted_UMI_count
cell_barcode,,,,,,,,,,,
AAACCCAAGAAACTAC-53,53,MRPS31,ENSG00000102738,P1P2,5261_MRPS31_P1P2_ENSG00000102738,MRPS31_-_41345123.23-P1P2|MRPS31_+_41345107.23...,0.051790,38405.0,0.065533,2.985979,12861.780273
AAACCCAAGAAGCCAC-51,51,LRRC37A3,ENSG00000176809,P1P2,4661_LRRC37A3_P1P2_ENSG00000176809,LRRC37A3_+_62915581.23-P1P2|LRRC37A3_-_6291539...,0.048614,12774.0,1.087875,0.721063,17715.509766
AAACCCAAGAAGCGAA-32,32,SRCAP,ENSG00000080603,P1P2,8397_SRCAP_P1P2_ENSG00000080603,SRCAP_-_30710672.23-P1P2|SRCAP_-_30710513.23-P1P2,0.049437,15353.0,0.571514,0.989524,15515.543945
AAACCCAAGAATACAC-44,44,WBP1,ENSG00000239779,P1P2,9793_WBP1_P1P2_ENSG00000239779,WBP1_-_74685599.23-P1P2|WBP1_-_74685547.23-P1P2,0.059373,18729.0,0.281205,1.321505,14172.480469
AAACCCAAGAATCGAT-43,43,RRP12,ENSG00000052749,P1P2,7561_RRP12_P1P2_ENSG00000052749,RRP12_+_99161057.23-P1P2|RRP12_-_99161036.23-P1P2,0.059395,11634.0,0.633577,0.735945,15808.244141


In [17]:
crispr_df = pd.read_csv(ref_dir / 'crispr' / 'hcrispri_all.csv')
crispr_df['sgID_clean'] = crispr_df['sgID'].astype(str).str.replace(',', '-', regex=False).str.strip()
crispr_df.head()

,sgID,gene,transcript,protospacer sequence,selection rank,predicted score,empirical score,off-target stringency,Sublibrary half,sgID_clean
0,A1BG_-_58858617.23-P1,A1BG,P1,GGAGACCCAGCGCTAACCAG,1.0,1.008816,NaN,0,Top5,A1BG_-_58858617.23-P1
1,A1BG_-_58858788.23-P1,A1BG,P1,GGGGCACCCAGGAGCGGTAG,2.0,0.901176,NaN,0,Top5,A1BG_-_58858788.23-P1
2,A1BG_+_58858964.23-P1,A1BG,P1,GCTCCGGGCGACGTGGAGTG,3.0,0.836188,NaN,0,Top5,A1BG_+_58858964.23-P1
3,A1BG_-_58858630.23-P1,A1BG,P1,GAACCAGGGGTGCCCAAGGG,4.0,0.827551,NaN,0,Top5,A1BG_-_58858630.23-P1
4,A1BG_+_58858549.23-P1,A1BG,P1,GGCGAGGAACCGCCCAGCAA,5.0,0.775395,NaN,0,Top5,A1BG_+_58858549.23-P1


In [18]:
id_to_seq = dict(zip(crispr_df['sgID_clean'].str.strip(), crispr_df['protospacer sequence'].str.strip()))


In [19]:
def get_sequences(dual_id_string):
    '''
    Parses 'GuideA_ID|GuideB_ID' and returns 'SeqA|SeqB'
    '''
    if pd.isna(dual_id_string) or dual_id_string == '':
        return None, None
        
    try:
        # The IDs in sgID_AB are separated by a pipe '|'
        parts = dual_id_string.split('|')

        def lookup(gid):
            gid = gid.strip()
            seq = id_to_seq.get(gid)
            
            if seq is None:
                gid_alt = gid.replace(',', '-')
                seq = id_to_seq.get(gid_alt, 'NOT_FOUND')
            
            return seq
        
        if len(parts) == 2:
            id_a, id_b = parts
            return lookup(id_a), lookup(id_b)
            
        elif len(parts) == 1:
            return lookup(parts[0]), 'NONE'
            
    except Exception as e:
        return 'ERROR', 'ERROR'

    return 'MISSING', 'MISSING'

In [25]:
dual_ids = obs_df['sgID_AB'].astype(str).tolist()
len(dual_ids)

247914

In [27]:
seq_a_list = []
seq_b_list = []
for dual_id in tqdm(dual_ids):
    sa, sb = get_sequences(dual_id)
    seq_a_list.append(sa)
    seq_b_list.append(sb)

100%|████████████████████████████████████████████████████████████████| 247914/247914 [00:00<00:00, 1962585.16it/s]


In [29]:
obs_df['guide_seq_a'] = seq_a_list
obs_df['guide_seq_b'] = seq_b_list

obs_df[['sgID_AB', 'guide_seq_a', 'guide_seq_b']].head()

,sgID_AB,guide_seq_a,guide_seq_b
cell_barcode,,,
AAACCCAAGAAACTAC-53,MRPS31_-_41345123.23-P1P2|MRPS31_+_41345107.23...,GGGTTGCCTGAGGACCTACC,GGTTGGCCCGGTAGGTCCTC
AAACCCAAGAAGCCAC-51,LRRC37A3_+_62915581.23-P1P2|LRRC37A3_-_6291539...,GCGTGACCGCGCTCCGGTAA,GCCTCATGTTGGAACCGAGG
AAACCCAAGAAGCGAA-32,SRCAP_-_30710672.23-P1P2|SRCAP_-_30710513.23-P1P2,GAGCATTTCCGGGCCTCCTG,GTCAGTCCGTCGGGAGGGCT
AAACCCAAGAATACAC-44,WBP1_-_74685599.23-P1P2|WBP1_-_74685547.23-P1P2,GTCTGCCCGGATGGAAGCTC,GGGGACGATGATGATTGGTA
AAACCCAAGAATCGAT-43,RRP12_+_99161057.23-P1P2|RRP12_-_99161036.23-P1P2,GGGCTACAGGGTATCCACGT,GCTTAAATGACCGGCTTCCA


In [32]:
a_mask = obs_df['guide_seq_a'].astype(str).str.contains(r'NOT_FOUND|MISSING|ERROR', na=False)
b_mask = obs_df['guide_seq_b'].astype(str).str.contains(r'NOT_FOUND|MISSING|ERROR', na=False)

len(set(obs_df[a_mask]['sgID_AB'].astype(str))), len(set(obs_df[b_mask]['sgID_AB'].astype(str)))


(0, 0)

#### k562 genome wide

In [33]:
ds_key = 'k562gw'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')

obs_df = ds_adata.obs
obs_df.head()

,batch,gene,gene_id,transcript,gene_transcript,guide_id,percent_mito,UMI_count,z_gemgroup_UMI,core_scale_factor,...,sex,age,perturbation,organism,perturbation_type,tissue_type,ncounts,ngenes,nperts,percent_ribo
cell_barcode,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGAAACCAT-157,157,CTSC,ENSG00000109861,P1P2,1946_CTSC_P1P2_ENSG00000109861,CTSC_-_88070848.23-P1P2|CTSC_-_88070918.23-P1P2,0.088177,14709.0,0.470687,1.051176,...,Female,53,CTSC,human,CRISPR,cell_line,14506.0,3440,1,0.282228
AAACCCAAGAAACCAT-207,207,CWC25,ENSG00000273559,P1P2,1973_CWC25_P1P2_ENSG00000273559,CWC25_+_36981555.23-P1P2|CWC25_+_36981567.23-P1P2,0.114342,16162.0,0.824790,1.074744,...,Female,53,CWC25,human,CRISPR,cell_line,15883.0,3872,1,0.253919
AAACCCAAGAAACCAT-29,29,PDE4DIP,ENSG00000178104,ENST00000313431.9,6168_PDE4DIP_ENST00000313431.9_ENSG00000178104,PDE4DIP_+_144932474.23-ENST00000313431.9|PDE4D...,0.107157,33297.0,2.627126,1.472444,...,Female,53,PDE4DIP,human,CRISPR,cell_line,32720.0,5452,1,0.234627
AAACCCAAGAAAGCGA-149,149,ZZEF1,ENSG00000074755,P1P2,10745_ZZEF1_P1P2_ENSG00000074755,ZZEF1_+_4046247.23-P1P2|ZZEF1_+_4046255.23-P1P2,0.143107,7435.0,0.918149,0.480401,...,Female,53,ZZEF1,human,CRISPR,cell_line,7336.0,2551,1,0.215513
AAACCCAAGAAATCCA-172,172,SNAPIN,ENSG00000143553,P1P2,8210_SNAPIN_P1P2_ENSG00000143553,SNAPIN_+_153631238.23-P1P2|SNAPIN_-_153631252....,0.130754,7755.0,-0.230920,0.695649,...,Female,53,SNAPIN,human,CRISPR,cell_line,7632.0,2562,1,0.235325


In [34]:
dual_ids = obs_df['guide_id'].astype(str).tolist()
len(dual_ids)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


1989578

In [35]:
seq_a_list = []
seq_b_list = []
for dual_id in tqdm(dual_ids):
    sa, sb = get_sequences(dual_id)
    seq_a_list.append(sa)
    seq_b_list.append(sb)

100%|██████████████████████████████████████████████████████████████| 1989578/1989578 [00:00<00:00, 2005178.44it/s]


In [37]:
obs_df['guide_seq_a'] = seq_a_list
obs_df['guide_seq_b'] = seq_b_list

obs_df[['guide_id', 'guide_seq_a', 'guide_seq_b']].head()

,guide_id,guide_seq_a,guide_seq_b
cell_barcode,,,
AAACCCAAGAAACCAT-157,CTSC_-_88070848.23-P1P2|CTSC_-_88070918.23-P1P2,GGCCCAGCACCCATGCTGCA,GTAGCGGTGAGTCCACCACG
AAACCCAAGAAACCAT-207,CWC25_+_36981555.23-P1P2|CWC25_+_36981567.23-P1P2,GTCTAGAGCGGTGGTGAAAC,GCCGGAAGTGACCTCTAGAG
AAACCCAAGAAACCAT-29,PDE4DIP_+_144932474.23-ENST00000313431.9|PDE4D...,GGGCGCGCAGCGGTACCTAG,GGAGCCGGAAGCCCGCTGAC
AAACCCAAGAAAGCGA-149,ZZEF1_+_4046247.23-P1P2|ZZEF1_+_4046255.23-P1P2,GGCTGCTGTCGGAGGTTGAC,GGCCCGCCAGCTGCTGTCGG
AAACCCAAGAAATCCA-172,SNAPIN_+_153631238.23-P1P2|SNAPIN_-_153631252....,GGTCCCTGCCCCCGATACAG,GGGTTCCGCCGCTGTATCGG


In [39]:
a_mask = obs_df['guide_seq_a'].astype(str).str.contains(r'NOT_FOUND|MISSING|ERROR', na=False)
b_mask = obs_df['guide_seq_b'].astype(str).str.contains(r'NOT_FOUND|MISSING|ERROR', na=False)

len(set(obs_df[a_mask]['guide_id'].astype(str))), len(set(obs_df[b_mask]['guide_id'].astype(str)))

(0, 0)

#### adamson

In [6]:
ds_key = 'adamson'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')

ad_df = ds_adata.obs
ad_df.head()


,perturbation,read count,UMI count,tissue_type,cell_line,cancer,disease,perturbation_type,celltype,organism,ncounts,ngenes,percent_mito,percent_ribo,nperts
cell_barcode,,,,,,,,,,,,,,,
AAACATACAAGATG,63(mod)_pBA580,282.0,8.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,8866.0,2914,4.917663,21.306112,2
AAACATACACCTAG,OST4_pDS353,331.0,7.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,13785.0,3818,4.468626,19.492201,2
AAACATACTTCCCG,SEC61A1_pDS031,285.0,10.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,7569.0,2616,5.060113,23.199894,2
AAACATTGAAACAG,EIF2B4_pDS491,1036.0,30.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,13834.0,3488,5.052769,28.733555,2
AAACATTGCAGCTA,SRPR_pDS482,863.0,25.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,15507.0,3620,4.514091,26.729864,2


In [7]:
cg_df = pd.read_csv(ref_dir / 'adamson/adamson_protospacer_sgrna.csv')
cg_df.head()

,Gene,Protospacer,Guide_ID (synonymous with sgGuide_ID),Perturb-seq_Vector_ID
0,AARS,GAGGGCGGCCTACCTCTCCT,NaN,pDS381
1,AMIGO3/GMPPB,GGAACGCGACACCGGGTAGA,NaN,pDS391
2,AMIGO3/GMPPB,GGGGCCAGCAGCCGTCTACC,NaN,pDS434
3,ARHGAP22,GGTCCGTCCGGAGCCAGGAG,NaN,pDS458
4,ASCC3,GACGCAAAGACGCACAGACC,NaN,pDS051


In [8]:
ad_df['match_key'] = ad_df['perturbation'].astype(str).apply(
    lambda x: x.rsplit('_', 1)[-1] if '_' in x else None
)

ad_df.head()

,perturbation,read count,UMI count,tissue_type,cell_line,cancer,disease,perturbation_type,celltype,organism,ncounts,ngenes,percent_mito,percent_ribo,nperts,match_key
cell_barcode,,,,,,,,,,,,,,,,
AAACATACAAGATG,63(mod)_pBA580,282.0,8.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,8866.0,2914,4.917663,21.306112,2,pBA580
AAACATACACCTAG,OST4_pDS353,331.0,7.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,13785.0,3818,4.468626,19.492201,2,pDS353
AAACATACTTCCCG,SEC61A1_pDS031,285.0,10.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,7569.0,2616,5.060113,23.199894,2,pDS031
AAACATTGAAACAG,EIF2B4_pDS491,1036.0,30.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,13834.0,3488,5.052769,28.733555,2,pDS491
AAACATTGCAGCTA,SRPR_pDS482,863.0,25.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,15507.0,3620,4.514091,26.729864,2,pDS482


In [9]:
cg_df['Guide_ID'] = cg_df['Perturb-seq_Vector_ID'].astype(str).str.strip()
dictionary_lookup = cg_df[['Guide_ID', 'Protospacer', 'Gene']].drop_duplicates(subset='Guide_ID')
dictionary_lookup.head()


,Guide_ID,Protospacer,Gene
0,pDS381,GAGGGCGGCCTACCTCTCCT,AARS
1,pDS391,GGAACGCGACACCGGGTAGA,AMIGO3/GMPPB
2,pDS434,GGGGCCAGCAGCCGTCTACC,AMIGO3/GMPPB
3,pDS458,GGTCCGTCCGGAGCCAGGAG,ARHGAP22
4,pDS051,GACGCAAAGACGCACAGACC,ASCC3


In [10]:
merged_df = ad_df.merge(
    dictionary_lookup,
    left_on='match_key',
    right_on='Guide_ID',
    how='left'
)

merged_df.head()

,perturbation,read count,UMI count,tissue_type,cell_line,cancer,disease,perturbation_type,celltype,organism,ncounts,ngenes,percent_mito,percent_ribo,nperts,match_key,Guide_ID,Protospacer,Gene
0,63(mod)_pBA580,282.0,8.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,8866.0,2914,4.917663,21.306112,2,pBA580,NaN,NaN,NaN
1,OST4_pDS353,331.0,7.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,13785.0,3818,4.468626,19.492201,2,pDS353,pDS353,GGCTTGTTCGCTGGTGGCGT,OST4
2,SEC61A1_pDS031,285.0,10.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,7569.0,2616,5.060113,23.199894,2,pDS031,pDS031,GCTGTGCAGTGGAACGCGCT,SEC61A1
3,EIF2B4_pDS491,1036.0,30.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,13834.0,3488,5.052769,28.733555,2,pDS491,pDS491,GCTGAGGGCGATGGCTGCTG,EIF2B4
4,SRPR_pDS482,863.0,25.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,15507.0,3620,4.514091,26.729864,2,pDS482,pDS482,GGCGAACGCGGCCTGAATTCC,SRPR


In [11]:
def get_protospacer_status(row):
    raw_pert = str(row['perturbation'])
    
    # CASE 1: Sequence Found
    if pd.notna(row['Protospacer']):
        return row['Protospacer']
    
    # CASE 2: It's a Control (Should not have a specific target sequence usually)
    if any(x in raw_pert.lower() for x in ['non-targeting', 'neg', 'ctrl', 'scramble','*','nan']):
        return "CONTROL_SEQ" # Or keep as None/NaN depending on your preference
        
    # CASE 3: Missing (The 22 cells you mentioned)
    return "MISSING_SEQUENCE"

In [12]:
merged_df['final_protospacer'] = merged_df.apply(get_protospacer_status, axis=1)
merged_df.head()

,perturbation,read count,UMI count,tissue_type,cell_line,cancer,disease,perturbation_type,celltype,organism,ncounts,ngenes,percent_mito,percent_ribo,nperts,match_key,Guide_ID,Protospacer,Gene,final_protospacer
0,63(mod)_pBA580,282.0,8.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,8866.0,2914,4.917663,21.306112,2,pBA580,NaN,NaN,NaN,MISSING_SEQUENCE
1,OST4_pDS353,331.0,7.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,13785.0,3818,4.468626,19.492201,2,pDS353,pDS353,GGCTTGTTCGCTGGTGGCGT,OST4,GGCTTGTTCGCTGGTGGCGT
2,SEC61A1_pDS031,285.0,10.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,7569.0,2616,5.060113,23.199894,2,pDS031,pDS031,GCTGTGCAGTGGAACGCGCT,SEC61A1,GCTGTGCAGTGGAACGCGCT
3,EIF2B4_pDS491,1036.0,30.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,13834.0,3488,5.052769,28.733555,2,pDS491,pDS491,GCTGAGGGCGATGGCTGCTG,EIF2B4,GCTGAGGGCGATGGCTGCTG
4,SRPR_pDS482,863.0,25.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,15507.0,3620,4.514091,26.729864,2,pDS482,pDS482,GGCGAACGCGGCCTGAATTCC,SRPR,GGCGAACGCGGCCTGAATTCC


In [13]:
def get_gene_target(row):
    # Trust Dictionary First
    if pd.notna(row['Gene']):
        return row['Gene']
        
    # Rescue from string if Dictionary failed
    raw_pert = str(row['perturbation'])
    if '_' in raw_pert and 'MISSING' in str(row['final_protospacer']):
        return raw_pert.rsplit('_', 1)[0] # "63(mod)"
    
    return 'Unknown'

merged_df['final_target'] = merged_df.apply(get_gene_target, axis=1)
merged_df.head()

,perturbation,read count,UMI count,tissue_type,cell_line,cancer,disease,perturbation_type,celltype,organism,...,ngenes,percent_mito,percent_ribo,nperts,match_key,Guide_ID,Protospacer,Gene,final_protospacer,final_target
0,63(mod)_pBA580,282.0,8.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,...,2914,4.917663,21.306112,2,pBA580,NaN,NaN,NaN,MISSING_SEQUENCE,63(mod)
1,OST4_pDS353,331.0,7.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,...,3818,4.468626,19.492201,2,pDS353,pDS353,GGCTTGTTCGCTGGTGGCGT,OST4,GGCTTGTTCGCTGGTGGCGT,OST4
2,SEC61A1_pDS031,285.0,10.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,...,2616,5.060113,23.199894,2,pDS031,pDS031,GCTGTGCAGTGGAACGCGCT,SEC61A1,GCTGTGCAGTGGAACGCGCT,SEC61A1
3,EIF2B4_pDS491,1036.0,30.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,...,3488,5.052769,28.733555,2,pDS491,pDS491,GCTGAGGGCGATGGCTGCTG,EIF2B4,GCTGAGGGCGATGGCTGCTG,EIF2B4
4,SRPR_pDS482,863.0,25.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,...,3620,4.514091,26.729864,2,pDS482,pDS482,GGCGAACGCGGCCTGAATTCC,SRPR,GGCGAACGCGGCCTGAATTCC,SRPR


In [14]:
set(merged_df[merged_df['final_protospacer'] == 'MISSING_SEQUENCE'].final_target)

{'62(mod)',
 '63(mod)',
 'ATF4',
 'ATF6',
 'C7orf26',
 'CCND3',
 'EIF2AK3',
 'ERN1',
 'GBF1',
 'Gal4-4(mod)',
 'IER3IP1',
 'PSMA1',
 'PSMD12',
 'STT3A',
 'XBP1',
 'YIPF5'}

### Norman

In [23]:
ds_key = 'norman'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')

obs_df = ds_adata.obs
obs_df.head()


,guide_id,read_count,UMI_count,coverage,gemgroup,good_coverage,number_of_cells,tissue_type,cell_line,cancer,disease,perturbation_type,celltype,organism,perturbation,nperts,ngenes,ncounts,percent_mito,percent_ribo
TTGAACGAGACTCGGA,ARID1A_NegCtrl0;ARID1A_NegCtrl0,28684,1809,15.856274,2,True,1,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,ARID1A,1,3079,15097.0,5.815725,33.569583
CGTTGGGGTGTTTGTG,BCORL1_NegCtrl0;BCORL1_NegCtrl0,18367,896,20.498884,7,True,1,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,BCORL1,1,2100,8551.0,4.104783,45.842592
GAACCTAAGTGTTAGA,FOSB_NegCtrl0;FOSB_NegCtrl0,16296,664,24.542169,6,True,1,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,FOSB,1,2772,10999.0,5.655060,17.801618
CCTTCCCTCCGTCATC,SET_KLF1;SET_KLF1,16262,850,19.131765,4,True,1,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,SET_KLF1,2,5385,38454.0,4.335050,38.165080
TCAATCTGTCTTTCAT,OSR2_NegCtrl0;OSR2_NegCtrl0,16057,1067,15.048735,2,True,2,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,OSR2,1,4869,27926.0,5.084867,32.317554


In [24]:
seq_map_df = pd.read_csv(ref_dir / 'norman/norman_sgrna.csv')
seq_map_df.head()

,number,gene_A,gene_B,protospacer_sequence_A,protospacer_sequence_B,GBC,Notes
0,1,AHR,NegCtrl0,GAGACGGAATGGAATCCAGA,GTCGCGCCCGCTCCAGGGAC,GAGTCGGACTCCGCCATG,NaN
1,2,ARID1A,NegCtrl0,GCCGCCTGGCAAACCCGGAG,GTCGCGCCCGCTCCAGGGAC,CACAGCATACTAGCGACC,NaN
2,3,ARRDC3,NegCtrl0,GGTACAGTAGGTGTAGAGCT,GTCGCGCCCGCTCCAGGGAC,AAGTTTGAGCGATGCCGT,NaN
3,4,ATL1,NegCtrl0,GAGTGCTCGGGCGGGCCGCT,GTCGCGCCCGCTCCAGGGAC,GCTAAGGGTTTGATGAGG,NaN
4,5,BAK1,NegCtrl0,GCAGGCAGGGCGGCTGTCAG,GTCGCGCCCGCTCCAGGGAC,TCAGACGTGGTGAGATCG,NaN


In [25]:
target_map_df = pd.read_csv(ref_dir / 'norman/norman_guideid_ensemble_id_map.csv')
target_map_df.head()

,guide_id,UMI_count,num_cells,first_target,first_id,control_first_expr,first_expr,fold_first_expr,second_target,second_id,control_second_expr,second_expr,fold_second_expr,ks_de,fitness,guide_UMI_count,guide_read_count,guide_coverage
0,AHR_FEV,8636.844697,264,AHR,ENSG00000106546,0.003264,0.613636,188.002841,FEV,ENSG00000163497,0.375493,5.174242,13.779864,3261,-0.577088,29.715909,644.553030,21.459278
1,AHR_KLF1,15603.354370,412,AHR,ENSG00000106546,0.003264,0.332524,101.877124,KLF1,ENSG00000105610,0.962464,4.614078,4.794025,379,-0.037193,45.427184,866.148058,19.279608
2,AHR_NegCtrl0,11797.929020,479,AHR,ENSG00000106546,0.003264,0.471816,144.552714,NegCtrl0,NaN,NaN,NaN,NaN,1345,-0.333858,46.966597,1025.187891,21.693670
3,ARID1A_NegCtrl0,10933.016480,182,ARID1A,ENSG00000117713,0.926697,0.961538,1.037598,NegCtrl0,NaN,NaN,NaN,NaN,1548,-0.319112,74.593407,1577.384615,22.406428
4,ARRDC3_NegCtrl0,13836.148150,405,ARRDC3,ENSG00000113369,0.052360,0.214815,4.102684,NegCtrl0,NaN,NaN,NaN,NaN,99,-0.126237,63.824691,1255.207407,19.590217


In [26]:
obs_df['clean_guide_id'] = obs_df['guide_id'].astype(str).apply(lambda x: x.split(';')[0])
obs_df.head()

,guide_id,read_count,UMI_count,coverage,gemgroup,good_coverage,number_of_cells,tissue_type,cell_line,cancer,...,perturbation_type,celltype,organism,perturbation,nperts,ngenes,ncounts,percent_mito,percent_ribo,clean_guide_id
TTGAACGAGACTCGGA,ARID1A_NegCtrl0;ARID1A_NegCtrl0,28684,1809,15.856274,2,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,ARID1A,1,3079,15097.0,5.815725,33.569583,ARID1A_NegCtrl0
CGTTGGGGTGTTTGTG,BCORL1_NegCtrl0;BCORL1_NegCtrl0,18367,896,20.498884,7,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,BCORL1,1,2100,8551.0,4.104783,45.842592,BCORL1_NegCtrl0
GAACCTAAGTGTTAGA,FOSB_NegCtrl0;FOSB_NegCtrl0,16296,664,24.542169,6,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,FOSB,1,2772,10999.0,5.655060,17.801618,FOSB_NegCtrl0
CCTTCCCTCCGTCATC,SET_KLF1;SET_KLF1,16262,850,19.131765,4,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,SET_KLF1,2,5385,38454.0,4.335050,38.165080,SET_KLF1
TCAATCTGTCTTTCAT,OSR2_NegCtrl0;OSR2_NegCtrl0,16057,1067,15.048735,2,True,2,cell_line,K562,True,...,CRISPR,lymphoblasts,human,OSR2,1,4869,27926.0,5.084867,32.317554,OSR2_NegCtrl0


In [28]:
seq_map_df['constructed_id'] = seq_map_df['gene_A'] + '_' + seq_map_df['gene_B']
seq_subset = seq_map_df[['constructed_id', 'protospacer_sequence_A', 'protospacer_sequence_B']]
seq_subset.head()

,constructed_id,protospacer_sequence_A,protospacer_sequence_B
0,AHR_NegCtrl0,GAGACGGAATGGAATCCAGA,GTCGCGCCCGCTCCAGGGAC
1,ARID1A_NegCtrl0,GCCGCCTGGCAAACCCGGAG,GTCGCGCCCGCTCCAGGGAC
2,ARRDC3_NegCtrl0,GGTACAGTAGGTGTAGAGCT,GTCGCGCCCGCTCCAGGGAC
3,ATL1_NegCtrl0,GAGTGCTCGGGCGGGCCGCT,GTCGCGCCCGCTCCAGGGAC
4,BAK1_NegCtrl0,GCAGGCAGGGCGGCTGTCAG,GTCGCGCCCGCTCCAGGGAC


In [29]:
target_subset = target_map_df[['guide_id', 'first_target', 'first_id', 'second_target', 'second_id']]
target_subset.head()

,guide_id,first_target,first_id,second_target,second_id
0,AHR_FEV,AHR,ENSG00000106546,FEV,ENSG00000163497
1,AHR_KLF1,AHR,ENSG00000106546,KLF1,ENSG00000105610
2,AHR_NegCtrl0,AHR,ENSG00000106546,NegCtrl0,NaN
3,ARID1A_NegCtrl0,ARID1A,ENSG00000117713,NegCtrl0,NaN
4,ARRDC3_NegCtrl0,ARRDC3,ENSG00000113369,NegCtrl0,NaN


In [31]:
merged_df = pd.merge(
        obs_df,
        target_subset,
        left_on='clean_guide_id',
        right_on='guide_id',
        how='left',
        suffixes=('', '_targetmap')
    )
final_df = pd.merge(
        merged_df,
        seq_subset,
        left_on='clean_guide_id',
        right_on='constructed_id',
        how='left'
    )
cols_to_drop = ['constructed_id', 'guide_id_targetmap', 'clean_guide_id']
final_df = final_df.drop(columns=[c for c in cols_to_drop if c in final_df.columns])
final_df.head()

,guide_id,read_count,UMI_count,coverage,gemgroup,good_coverage,number_of_cells,tissue_type,cell_line,cancer,...,ngenes,ncounts,percent_mito,percent_ribo,first_target,first_id,second_target,second_id,protospacer_sequence_A,protospacer_sequence_B
0,ARID1A_NegCtrl0;ARID1A_NegCtrl0,28684,1809,15.856274,2,True,1,cell_line,K562,True,...,3079,15097.0,5.815725,33.569583,ARID1A,ENSG00000117713,NegCtrl0,NaN,GCCGCCTGGCAAACCCGGAG,GTCGCGCCCGCTCCAGGGAC
1,BCORL1_NegCtrl0;BCORL1_NegCtrl0,18367,896,20.498884,7,True,1,cell_line,K562,True,...,2100,8551.0,4.104783,45.842592,BCORL1,ENSG00000085185,NegCtrl0,NaN,GGATCGCTGAGAGGACCGAG,GTCGCGCCCGCTCCAGGGAC
2,FOSB_NegCtrl0;FOSB_NegCtrl0,16296,664,24.542169,6,True,1,cell_line,K562,True,...,2772,10999.0,5.655060,17.801618,FOSB,ENSG00000125740,NegCtrl0,NaN,GGATCCCGGCCCCGCCTTCC,GTCGCGCCCGCTCCAGGGAC
3,SET_KLF1;SET_KLF1,16262,850,19.131765,4,True,1,cell_line,K562,True,...,5385,38454.0,4.335050,38.165080,SET,ENSG00000119335,KLF1,ENSG00000105610,GCACAGGGCCCGGCGAGAGG,GGGGCTGTGGAGCCTCAATC
4,OSR2_NegCtrl0;OSR2_NegCtrl0,16057,1067,15.048735,2,True,2,cell_line,K562,True,...,4869,27926.0,5.084867,32.317554,OSR2,ENSG00000164920,NegCtrl0,NaN,GCGCTGAGGGCCCCGCGCGG,GTCGCGCCCGCTCCAGGGAC


### sciplex

In [53]:
ds_key = 'sciplex'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')

obs_df = ds_adata.obs
obs_df.head()


,ncounts,well,plate,cell_line,replicate,time,dose_value,pathway_level_1,pathway_level_2,perturbation,target,pathway,dose_unit,celltype,disease,cancer,tissue_type,organism,perturbation_type
A01_E09_RT_BC_100_Lig_BC_147,2957,plate6_A9,plate44,MCF7,rep2,24.0,10000.0,Tyrosine kinase signaling,RTK activity,TAK-901,Aurora Kinase,Cell Cycle,nM,mammary epithelial cells,breast adenocarcinoma,True,cell_line,human,drug
A01_E09_RT_BC_100_Lig_BC_186,1528,plate8_H3,plate46,MCF7,rep2,24.0,10.0,Tyrosine kinase signaling,RTK activity,AG-490 (Tyrphostin B42),EGFR,Protein Tyrosine Kinase,nM,mammary epithelial cells,breast adenocarcinoma,True,cell_line,human,drug
A01_E09_RT_BC_100_Lig_BC_196,1881,plate3_C2,plate41,MCF7,rep2,24.0,1000.0,Epigenetic regulation,Histone deacetylation,Abexinostat (PCI-24781),HDAC,Cytoskeletal Signaling,nM,mammary epithelial cells,breast adenocarcinoma,True,cell_line,human,drug
A01_E09_RT_BC_100_Lig_BC_213,1700,plate9_E3,plate51,A549,rep2,72.0,1000.0,Cell cycle regulation,Aurora kinase activity,Alisertib (MLN8237),Aurora Kinase,Cell Cycle,nM,alveolar basal epithelial cells,lung adenocarcinoma,True,cell_line,human,drug
A01_E09_RT_BC_100_Lig_BC_220,1430,plate8_H10,plate30,K562,rep2,24.0,10000.0,DNA damage & DNA repair,Alkylating agent,Busulfan,DNA alkylator,DNA Damage,nM,lymphoblasts,chronic myelogenous leukemia,True,cell_line,human,drug


In [67]:
def get_smiles_robust(raw_name):
    clean_parts = re.split(r'[()\[\]]?', raw_name)
    candidates = [raw_name] + [p.strip() for p in clean_parts if p.strip()]
    
    # Remove duplicates while preserving order
    candidates = list(dict.fromkeys(candidates))

    for candidate in candidates:
        try:
            compounds = pcp.get_compounds(candidate, 'name')
            if compounds:
                return compounds[0].isomeric_smiles
        except Exception:
            continue
            
    return None

In [68]:
drugs = [i for i in set(obs_df.perturbation) if str(i) not in ['control','nan']]
results = []

In [69]:
for d in drugs:
    smiles = get_smiles_robust(d)
    if smiles:
        results.append({'drug_name': d, 'smiles': smiles})
    else:
        print(f'FAILED {d}')
    
    

In [70]:
pd.DataFrame(results)

,drug_name,smiles
0,Regorafenib (BAY 73-4506),CNC(=O)C1=NC=CC(=C1)OC2=CC(=C(C=C2)NC(=O)NC3=C...
1,GSK J1,C1CN(CCC2=CC=CC=C21)C3=NC(=NC(=C3)NCCC(=O)O)C4...
2,Hesperadin,CCS(=O)(=O)NC1=CC2=C(C=C1)NC(=C2C(=NC3=CC=C(C=...
3,TAK-901,CCS(=O)(=O)C1=CC=CC(=C1)C2=CC(=C(C3=C2C4=C(N3)...
4,AR-42,CC(C)[C@@H](C1=CC=CC=C1)C(=O)NC2=CC=C(C=C2)C(=...
...,...,...
183,Divalproex Sodium,CCCC(CCC)C(=O)O.CCCC(CCC)C(=O)[O-].[Na+]
184,"Patupilone (EPO906, Epothilone B)",C[C@H]1CCC[C@@]2([C@@H](O2)C[C@H](OC(=O)C[C@@H...
185,Prednisone,C[C@]12CC(=O)[C@H]3[C@H]([C@@H]1CC[C@@]2(C(=O)...
186,JNJ-7706621,C1=CC(=C(C(=C1)F)C(=O)N2C(=NC(=N2)NC3=CC=C(C=C...


In [71]:
final_df = pd.merge(
        obs_df,
        pd.DataFrame(results),
        left_on='perturbation',
        right_on='drug_name',
        how='left'
    )

final_df.head()

,ncounts,well,plate,cell_line,replicate,time,dose_value,pathway_level_1,pathway_level_2,perturbation,...,pathway,dose_unit,celltype,disease,cancer,tissue_type,organism,perturbation_type,drug_name,smiles
0,2957,plate6_A9,plate44,MCF7,rep2,24.0,10000.0,Tyrosine kinase signaling,RTK activity,TAK-901,...,Cell Cycle,nM,mammary epithelial cells,breast adenocarcinoma,True,cell_line,human,drug,TAK-901,CCS(=O)(=O)C1=CC=CC(=C1)C2=CC(=C(C3=C2C4=C(N3)...
1,1528,plate8_H3,plate46,MCF7,rep2,24.0,10.0,Tyrosine kinase signaling,RTK activity,AG-490 (Tyrphostin B42),...,Protein Tyrosine Kinase,nM,mammary epithelial cells,breast adenocarcinoma,True,cell_line,human,drug,AG-490 (Tyrphostin B42),C1=CC=C(C=C1)CNC(=O)/C(=C/C2=CC(=C(C=C2)O)O)/C#N
2,1881,plate3_C2,plate41,MCF7,rep2,24.0,1000.0,Epigenetic regulation,Histone deacetylation,Abexinostat (PCI-24781),...,Cytoskeletal Signaling,nM,mammary epithelial cells,breast adenocarcinoma,True,cell_line,human,drug,Abexinostat (PCI-24781),CN(C)CC1=C(OC2=CC=CC=C21)C(=O)NCCOC3=CC=C(C=C3...
3,1700,plate9_E3,plate51,A549,rep2,72.0,1000.0,Cell cycle regulation,Aurora kinase activity,Alisertib (MLN8237),...,Cell Cycle,nM,alveolar basal epithelial cells,lung adenocarcinoma,True,cell_line,human,drug,Alisertib (MLN8237),COC1=C(C(=CC=C1)F)C2=NCC3=CN=C(N=C3C4=C2C=C(C=...
4,1430,plate8_H10,plate30,K562,rep2,24.0,10000.0,DNA damage & DNA repair,Alkylating agent,Busulfan,...,DNA Damage,nM,lymphoblasts,chronic myelogenous leukemia,True,cell_line,human,drug,Busulfan,CS(=O)(=O)OCCCCOS(=O)(=O)C
